In [1]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")
print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [3]:
# Drop rows with no CustomerID 
df = df.dropna(subset=['CustomerID'])

# Change CustomerID from float to int to remove .0 then to string 
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

# Remove cancelled orders and bad rows with negative quantity or price
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Add Revenue column 
df['Revenue'] = df['Quantity'] * df['UnitPrice']


print(df.shape)
df.head()

(397884, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [4]:
#check countries
print(df['Country'].value_counts())

Country
United Kingdom          354321
Germany                   9040
France                    8341
EIRE                      7236
Spain                     2484
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1462
Australia                 1182
Norway                    1071
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     248
Unspecified                244
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         57
Lebanon                     45


In [ ]:
#top 10 revenue countries
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
print(country_revenue) 


Country
United Kingdom    7308391.554
Netherlands        285446.340
EIRE               265545.900
Germany            228867.140
France             209024.050
Australia          138521.310
Spain               61577.110
Switzerland         56443.950
Belgium             41196.340
Sweden              38378.330
Name: Revenue, dtype: float64


In [7]:
#top 10 best selling products
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products)

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64


In [8]:
#How does revenue trend month by month?
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly_revenue = df.groupby('Month')['Revenue'].sum()
print(monthly_revenue)

Month
2010-12     572713.890
2011-01     569445.040
2011-02     447137.350
2011-03     595500.760
2011-04     469200.361
2011-05     678594.560
2011-06     661213.690
2011-07     600091.011
2011-08     645343.900
2011-09     952838.382
2011-10    1039318.790
2011-11    1161817.380
2011-12     518192.790
Freq: M, Name: Revenue, dtype: float64


In [9]:
#prices of the top products
top_products = df.groupby('Description').agg(
    TotalQuantity=('Quantity', 'sum'),
    AvgPrice=('UnitPrice', 'mean')
).sort_values('TotalQuantity', ascending=False).head(10)

print(top_products)

                                    TotalQuantity  AvgPrice
Description                                                
PAPER CRAFT , LITTLE BIRDIE                 80995  2.080000
MEDIUM CERAMIC TOP STORAGE JAR              77916  1.220303
WORLD WAR 2 GLIDERS ASSTD DESIGNS           54415  0.292600
JUMBO BAG RED RETROSPOT                     46181  2.015878
WHITE HANGING HEART T-LIGHT HOLDER          36725  2.893107
ASSORTED COLOUR BIRD ORNAMENT               35362  1.680795
PACK OF 72 RETROSPOT CAKE CASES             33693  0.548212
POPCORN HOLDER                              30931  0.843272
RABBIT NIGHT LIGHT                          27202  2.013943
MINI PAINT SET VINTAGE                      26076  0.656523


In [13]:
#checking mini paint set vitange as it appears to have an abnormally low price
df[df['Description'] == 'MINI PAINT SET VINTAGE'][['UnitPrice', 'Quantity']].describe()

,UnitPrice,Quantity
count,325.000000,325.000000
mean,0.656523,80.233846
std,0.111219,153.494965
min,0.550000,1.000000
25%,0.650000,36.000000
50%,0.650000,36.000000
75%,0.650000,36.000000
max,1.630000,1394.000000


In [11]:
df[df['Description'].str.contains('MINI PAINT', na=False)]['Description'].unique()

array(['MINI PAINT SET VINTAGE ', 'MINI PAINTED GARDEN DECORATION '],
      dtype=object)

In [12]:
df['Description'] = df['Description'].str.strip()

In [14]:
df.to_csv("clean_retail.csv", index=False)
print("saved")

saved


In [15]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

today = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print(today)

2011-12-10 12:50:00


In [16]:
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (today - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

print(rfm.shape)
rfm.head()

(4338, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [17]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4])

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm.head())

  CustomerID  Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Score
0      12346      326          1  77183.60       1       1       4       114
1      12347        2          7   4310.00       4       4       4       444
2      12348       75          4   1797.24       2       3       4       234
3      12349       19          1   1757.55       3       1       4       314
4      12350      310          1    334.40       1       1       2       112


In [18]:
def assign_segment(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
 
    
    # Champions: recent, frequent, high spend
    if r >= 3 and f >= 3 and m >= 3:
        return 'Champion'
    # Loyal Customers: recent and frequent regardless of spend
    # e.g. 431, 341, 332 — keeps coming back even if not big spenders
    elif r >= 3 and f >= 3:
        return 'Loyal Customer'
    # At Risk: used to buy often and spend a lot but gone quiet
    elif f >= 3 and m >= 3 and r < 3:
        return 'At Risk'
    # Recent Customers: just arrived, haven't bought much yet
    elif r >= 3 and f <= 2:
        return 'Recent Customer'
    # Lost: low recency and only bought once
    elif r == 1 and f == 1:
        return 'Lost'
    # Everyone else: middle ground
    else:
        return 'Needs Attention'

# Apply the function row by row
rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Count how many customers in each segment
rfm['Segment'].value_counts()   

Segment
Champion           1319
Needs Attention    1183
Recent Customer     665
Lost                519
At Risk             448
Loyal Customer      204
Name: count, dtype: int64

In [19]:
rfm.to_csv("rfm_segments.csv", index=False)
print("saved!")

saved!


In [20]:
print(df['InvoiceDate'].min())
print(df['InvoiceDate'].max())


2010-12-01 08:26:00
2011-12-09 12:50:00
